# 26 — Run LIANA global main (memory-safe canonical run)

**Purpose:** Run the canonical global LIANA ligand–receptor analysis on the fixed Notebook 25 main-cell membership.

Historical references:
- `24_run_LIANA_global_main_HVG8000_r0_6.ipynb`
- `24b_fix_LIANA_global_main_labels_and_axes_HVG8000_r0_6_v2_fixed_source_clean.ipynb`

This notebook preserves the historical analytical settings while fixing the upstream label semantics through the canonical Notebook 25 outputs.

### Memory-safety design
- The full 351,427-cell AnnData is **not** loaded into RAM.
- Only the selected 184,066 main LIANA cells are read from the `counts` layer.
- The counts matrix is streamed from the H5AD CSR layer in row chunks.
- Only a minimal AnnData (`X`, `obs[groupby]`, `var_names`) is constructed.
- Normalization/log1p is performed in-place on sparse `float32` data.
- `N_JOBS=1` is used as an execution-only memory-safety setting; it does not change the analytical definition of the LIANA run.
- The large expression object is deleted immediately after LIANA results are copied.

Run **top to bottom in a fresh kernel**. Do not rerun upstream notebooks.


In [1]:
# Environment
import sys, os, gc, platform, warnings
from pathlib import Path
from datetime import datetime

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Started:", datetime.now().isoformat(timespec="seconds"))


Python: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0
Started: 2026-09-07T16:21:20


In [2]:
# Imports
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad
import h5py
import matplotlib.pyplot as plt

try:
    import psutil
    PSUTIL_AVAILABLE = True
except Exception:
    PSUTIL_AVAILABLE = False

try:
    import liana as li
    LIANA_AVAILABLE = True
    LIANA_VERSION = str(getattr(li, "__version__", "unknown_version"))
    print("[OK] liana:", LIANA_VERSION)
except Exception as e:
    LIANA_AVAILABLE = False
    LIANA_VERSION = "unavailable"
    print("[ERROR] Could not import liana:", repr(e))


[OK] liana: 1.7.1


In [3]:
# Fixed paths and canonical configuration
def _find_repository_root(start=None):
    # Resolve repository root only; analytical input filenames remain fixed.
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()

METADATA_DIR = PROJECT_DIR / "metadata"
FIG_DIR = PROJECT_DIR / "figures" / "LIANA_HVG8000_r0_6"
OUT_DIR = PROJECT_DIR / "data" / "processed" / "integrated"

METADATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

INPUT_H5AD = OUT_DIR / "GBM_core_4datasets_final_annotation_malignant_states.h5ad"
MAIN_CELLS_TSV = METADATA_DIR / "LIANA_prepare_HVG8000_r0_6_main_cells.tsv"
INCLUSION_TSV = METADATA_DIR / "LIANA_prepare_HVG8000_r0_6_celltype_inclusion_decisions.tsv"

OUT_PREFIX = "LIANA_global_main_HVG8000_r0_6"

# Historical/canonical analytical settings
GROUPBY = "liana_celltype_global_main"
COUNTS_LAYER = "counts"
RESOURCE_NAME = "consensus"
EXPR_PROP = 0.10
MIN_CELLS = 50
N_PERMS = None
SEED = 1337

# Memory-safety execution setting.
# Historical notebook used n_jobs=4; n_jobs=1 avoids process-level RAM duplication on Windows.
N_JOBS = 1
HISTORICAL_N_JOBS = 4
H5_ROW_CHUNK = 512
NORMALIZATION_TARGET_SUM = 1e4

# Fixed output selection semantics
EXPECTED_PRIMARY_RANK = "magnitude_rank"
RANK_THRESHOLD = 0.05
TOP_N_GLOBAL = 1000
TOP_N_PER_SOURCE_TARGET = 30

# Fixed upstream checkpoints from canonical Notebook 25
EXPECTED_INPUT_N_OBS = 351_427
EXPECTED_INPUT_N_VARS = 15_176
EXPECTED_MAIN_N_CELLS = 184_066
EXPECTED_MAIN_N_CELLTYPES = 9
EXPECTED_NORMALIZED_NNZ = 431_978_608
EXPECTED_LIANA_VERSION = "1.7.1"

EXPECTED_MAIN_COUNTS = {
    "Malignant_OPC_NPC_like_malignant": 61_646,
    "Malignant_cycling_candidate": 38_634,
    "Malignant_hypoxia_stress_malignant": 32_896,
    "Malignant_AC_like_reactive_malignant": 22_525,
    "Oligodendrocyte_myelinating": 14_809,
    "Malignant_NPC_neuronal_like_malignant": 6_183,
    "Mural_Pericyte_Fibroblast": 3_566,
    "Endothelial": 2_640,
    "B_Plasma": 1_167,
}

# Historical result checkpoints: audit only, never used to manufacture/truncate results.
HISTORICAL_RESULT_CHECKPOINTS = {
    "n_liana_results": 23_797,
    "n_prioritized": 1_384,
    "n_unique_lr_pairs": 264,
    "n_source_target_pairs": 81,
}

EXPECTED_COMMUNICATION_AXES = {
    "malignant_or_cycling_to_malignant_or_cycling",
    "malignant_or_cycling_to_non_malignant_reference_like",
    "non_malignant_reference_like_to_malignant_or_cycling",
    "non_malignant_reference_like_to_non_malignant_reference_like",
}

print("PROJECT_DIR:", PROJECT_DIR)
print("INPUT_H5AD:", INPUT_H5AD)
print("MAIN_CELLS_TSV:", MAIN_CELLS_TSV)
print("INCLUSION_TSV:", INCLUSION_TSV)


PROJECT_DIR: G:\Repository\glioma-cnv-single-cell-atlas
INPUT_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_final_annotation_malignant_states.h5ad
MAIN_CELLS_TSV: G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_prepare_HVG8000_r0_6_main_cells.tsv
INCLUSION_TSV: G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_prepare_HVG8000_r0_6_celltype_inclusion_decisions.tsv


In [4]:
# Helpers
def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print("[WRITE]", path, df.shape)
    return path

def savefig(path, dpi=180):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print("[FIG]", path)
    return path

def safe_read_tsv(path, **kwargs):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(str(path))
    return pd.read_csv(path, sep="\t", **kwargs)

def strict_bool_series(s, name):
    # Parse only explicit booleans/0/1; never use Python truthiness on strings.
    if pd.api.types.is_bool_dtype(s):
        return s.astype(bool)
    out = pd.Series(index=s.index, dtype="boolean")
    for idx, val in s.items():
        if pd.isna(val):
            raise ValueError(f"{name}: NA boolean value at row {idx}")
        if isinstance(val, (bool, np.bool_)):
            out.loc[idx] = bool(val)
            continue
        if isinstance(val, (int, np.integer)) and val in (0, 1):
            out.loc[idx] = bool(val)
            continue
        sval = str(val).strip().lower()
        if sval == "true":
            out.loc[idx] = True
        elif sval == "false":
            out.loc[idx] = False
        elif sval in {"0", "1"}:
            out.loc[idx] = (sval == "1")
        else:
            raise ValueError(f"{name}: invalid boolean token {val!r} at row {idx}")
    return out.astype(bool)

def available_ram_gb():
    if not PSUTIL_AVAILABLE:
        return np.nan
    return psutil.virtual_memory().available / (1024 ** 3)

def _decode_attr(x):
    if isinstance(x, bytes):
        return x.decode()
    return str(x)


def read_h5ad_axis_index(h5ad_path, axis):
    # Read only obs/var index strings directly from HDF5; never instantiate the full AnnData.
    with h5py.File(h5ad_path, "r") as h5:
        if axis not in h5:
            raise KeyError(f"H5AD axis group {axis!r} is missing.")
        g = h5[axis]
        idx_key = _decode_attr(g.attrs.get("_index", "_index"))
        if idx_key not in g:
            raise KeyError(f"H5AD {axis!r} index dataset {idx_key!r} is missing.")
        values = g[idx_key][:]
    decoded = [
        x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x)
        for x in values
    ]
    return pd.Index(decoded)

def inspect_csr_layer(h5ad_path, layer_name, row_positions):
    # Inspect selected-row CSR footprint without loading expression values.
    row_positions = np.asarray(row_positions, dtype=np.int64)
    if row_positions.ndim != 1 or len(row_positions) == 0:
        raise ValueError("row_positions must be a non-empty 1D array.")
    if np.any(np.diff(row_positions) <= 0):
        raise ValueError("row_positions must be strictly increasing and unique.")

    with h5py.File(h5ad_path, "r") as h5:
        key = f"layers/{layer_name}"
        if key not in h5:
            raise KeyError(f"Required layer {layer_name!r} not found at {key!r}.")
        g = h5[key]
        enc = _decode_attr(g.attrs.get("encoding-type", ""))
        if enc != "csr_matrix":
            raise RuntimeError(
                f"Memory-safe loader requires CSR-encoded {layer_name!r}; "
                f"found encoding-type={enc!r}. Stop rather than loading the full object."
            )
        shape = tuple(int(x) for x in g.attrs["shape"])
        if row_positions[0] < 0 or row_positions[-1] >= shape[0]:
            raise IndexError("Selected row positions are outside counts-layer bounds.")
        indptr = np.asarray(g["indptr"][:], dtype=np.int64)
        row_nnz = indptr[row_positions + 1] - indptr[row_positions]
        selected_nnz = int(row_nnz.sum())
        indices_dtype = np.dtype(g["indices"].dtype)
        source_data_dtype = np.dtype(g["data"].dtype)

    # Output data are written directly as float32 to avoid a second whole-matrix cast.
    target_bytes = (
        selected_nnz * (np.dtype(np.float32).itemsize + indices_dtype.itemsize)
        + (len(row_positions) + 1) * np.dtype(np.int64).itemsize
    )
    return {
        "shape": shape,
        "encoding_type": enc,
        "selected_nnz": selected_nnz,
        "source_data_dtype": str(source_data_dtype),
        "indices_dtype": str(indices_dtype),
        "target_csr_gb": target_bytes / (1024 ** 3),
    }

def load_selected_counts_csr(h5ad_path, layer_name, row_positions, chunk_rows=512):
    # Stream selected CSR rows from H5AD into one float32 CSR matrix.
    row_positions = np.asarray(row_positions, dtype=np.int64)
    meta = inspect_csr_layer(h5ad_path, layer_name, row_positions)
    n_rows_src, n_vars = meta["shape"]

    selected_mask = np.zeros(n_rows_src, dtype=bool)
    selected_mask[row_positions] = True

    with h5py.File(h5ad_path, "r") as h5:
        g = h5[f"layers/{layer_name}"]
        indptr_src = np.asarray(g["indptr"][:], dtype=np.int64)

        row_nnz = indptr_src[row_positions + 1] - indptr_src[row_positions]
        out_indptr = np.empty(len(row_positions) + 1, dtype=np.int64)
        out_indptr[0] = 0
        np.cumsum(row_nnz, out=out_indptr[1:])
        total_nnz = int(out_indptr[-1])

        out_data = np.empty(total_nnz, dtype=np.float32)
        out_indices = np.empty(total_nnz, dtype=g["indices"].dtype)

        cursor = 0
        for start in range(0, n_rows_src, chunk_rows):
            stop = min(start + chunk_rows, n_rows_src)
            rel = np.flatnonzero(selected_mask[start:stop])
            if len(rel) == 0:
                continue

            p0 = int(indptr_src[start])
            p1 = int(indptr_src[stop])

            chunk_data = np.asarray(g["data"][p0:p1])
            chunk_indices = np.asarray(g["indices"][p0:p1])
            chunk_indptr = indptr_src[start:stop + 1] - p0

            chunk = sp.csr_matrix(
                (chunk_data, chunk_indices, chunk_indptr),
                shape=(stop - start, n_vars),
            )
            sub = chunk[rel]

            n = int(sub.nnz)
            out_data[cursor:cursor + n] = sub.data
            out_indices[cursor:cursor + n] = sub.indices
            cursor += n

            del chunk_data, chunk_indices, chunk_indptr, chunk, sub

        if cursor != total_nnz:
            raise RuntimeError(f"CSR extraction mismatch: copied {cursor} != expected {total_nnz}")

    X = sp.csr_matrix(
        (out_data, out_indices, out_indptr),
        shape=(len(row_positions), n_vars),
        copy=False,
    )
    if not X.has_sorted_indices:
        X.sort_indices()

    del selected_mask
    gc.collect()
    return X, meta

def normalize_log1p_csr_inplace(X, target_sum=1e4, row_chunk=2048):
    # Equivalent library-size normalization + log1p, without duplicating the CSR matrix.
    if not sp.isspmatrix_csr(X):
        raise TypeError("Expected CSR matrix.")
    if X.dtype != np.float32:
        raise TypeError(f"Expected float32 CSR data, found {X.dtype}")

    row_sums = np.asarray(X.sum(axis=1)).ravel().astype(np.float64, copy=False)
    if np.any(~np.isfinite(row_sums)) or np.any(row_sums <= 0):
        bad = int(np.sum((~np.isfinite(row_sums)) | (row_sums <= 0)))
        raise ValueError(f"Found {bad} cells with non-finite or non-positive total counts.")

    scales = (float(target_sum) / row_sums).astype(np.float32)

    # Chunked repeat keeps temporary memory bounded.
    for r0 in range(0, X.shape[0], row_chunk):
        r1 = min(r0 + row_chunk, X.shape[0])
        p0 = int(X.indptr[r0])
        p1 = int(X.indptr[r1])
        repeats = np.diff(X.indptr[r0:r1 + 1])
        local_scale = np.repeat(scales[r0:r1], repeats)
        X.data[p0:p1] *= local_scale
        del local_scale

    np.log1p(X.data, out=X.data)
    del row_sums, scales
    gc.collect()
    return X

def broad_group(x):
    x = str(x)
    if x.startswith("Malignant_"):
        return "malignant_or_cycling"
    return "non_malignant_reference_like"

def label_axis(row):
    src = str(row["source"])
    tgt = str(row["target"])
    src_mal = src.startswith("Malignant_")
    tgt_mal = tgt.startswith("Malignant_")
    if src_mal and not tgt_mal:
        return "malignant_to_microenvironment"
    if not src_mal and tgt_mal:
        return "microenvironment_to_malignant"
    if src_mal and tgt_mal:
        return "malignant_state_to_malignant_state"
    return "microenvironment_to_microenvironment"

def target_lineage(t):
    t = str(t).lower()
    if "endothelial" in t:
        return "endothelial"
    if "mural" in t or "pericyte" in t or "fibroblast" in t:
        return "mural_pericyte_fibroblast"
    if "oligodendrocyte" in t:
        return "oligodendrocyte_reference"
    if "b_plasma" in t or "plasma" in t:
        return "B_plasma"
    if "malignant" in t or "cycling" in t:
        return "malignant_state"
    return "other"


In [5]:
# Readiness and version provenance — fail before loading expression if anything is wrong
readiness_records = []
for name, path in [
    ("input_h5ad", INPUT_H5AD),
    ("main_cells_tsv", MAIN_CELLS_TSV),
    ("inclusion_decisions_tsv", INCLUSION_TSV),
]:
    readiness_records.append({
        "item": name,
        "path": str(path),
        "exists": Path(path).exists(),
        "size_mb": round(Path(path).stat().st_size / 1024 / 1024, 3) if Path(path).exists() else np.nan,
    })

readiness_records.extend([
    {"item": "liana_available", "path": "", "exists": bool(LIANA_AVAILABLE), "size_mb": np.nan},
    {"item": "liana_version", "path": LIANA_VERSION, "exists": LIANA_VERSION == EXPECTED_LIANA_VERSION, "size_mb": np.nan},
])

readiness = pd.DataFrame(readiness_records)
write_tsv_replace(readiness, METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv")
display(readiness)

if not LIANA_AVAILABLE:
    raise ImportError("liana is not available in this environment.")
if LIANA_VERSION != EXPECTED_LIANA_VERSION:
    raise RuntimeError(
        f"Canonical historical reference uses liana=={EXPECTED_LIANA_VERSION}; "
        f"current environment has {LIANA_VERSION}. Stop before rerunning."
    )
for p in (INPUT_H5AD, MAIN_CELLS_TSV, INCLUSION_TSV):
    if not Path(p).exists():
        raise FileNotFoundError(str(p))


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_readiness.tsv (5, 4)


,item,path,exists,size_mb
0,input_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,True,14469.275
1,main_cells_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,45.386
2,inclusion_decisions_tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
3,liana_available,,True,NaN
4,liana_version,1.7.1,True,NaN


In [6]:
# Strict Notebook 25 membership validation
main_cells = safe_read_tsv(MAIN_CELLS_TSV)
decision = safe_read_tsv(INCLUSION_TSV)

required_main_cols = {"cell_id", "liana_celltype_main", "liana_include_main_global"}
required_decision_cols = {"liana_celltype", "include_main_global"}

missing_main = required_main_cols - set(main_cells.columns)
missing_decision = required_decision_cols - set(decision.columns)
if missing_main:
    raise ValueError(f"Missing canonical main-cell columns: {sorted(missing_main)}")
if missing_decision:
    raise ValueError(f"Missing canonical inclusion-decision columns: {sorted(missing_decision)}")

if main_cells["cell_id"].isna().any() or main_cells["cell_id"].duplicated().any():
    raise ValueError("main_cells contains NA or duplicate cell_id values.")
if decision["liana_celltype"].isna().any() or decision["liana_celltype"].duplicated().any():
    raise ValueError("inclusion_decisions contains NA or duplicate liana_celltype values.")

main_cells["_include"] = strict_bool_series(
    main_cells["liana_include_main_global"], "liana_include_main_global"
)
decision["_include"] = strict_bool_series(
    decision["include_main_global"], "include_main_global"
)

included_types = set(decision.loc[decision["_include"], "liana_celltype"].astype(str))
expected_types = set(EXPECTED_MAIN_COUNTS)

if included_types != expected_types:
    raise AssertionError(
        "Canonical global-inclusion cell types differ from expected Notebook 25 membership.\n"
        f"Expected only: {sorted(expected_types)}\nObserved: {sorted(included_types)}"
    )

main_cells = main_cells.loc[main_cells["_include"]].copy()
main_cells["liana_celltype_main"] = main_cells["liana_celltype_main"].astype(str)

bad_old = main_cells["liana_celltype_main"].str.contains("Malignant_non_malignant", regex=False)
bad_review = main_cells["liana_celltype_main"].str.startswith("Review_cluster_")
bad_mast = main_cells["liana_celltype_main"].eq("Mast_cell")
if bad_old.any() or bad_review.any() or bad_mast.any():
    raise AssertionError(
        f"Forbidden main-global labels found: old_bug={int(bad_old.sum())}, "
        f"review={int(bad_review.sum())}, mast={int(bad_mast.sum())}"
    )

observed_counts = main_cells["liana_celltype_main"].value_counts().sort_index()
expected_counts = pd.Series(EXPECTED_MAIN_COUNTS, name="expected").sort_index()
count_audit = pd.DataFrame({
    "liana_celltype": expected_counts.index,
    "expected_n_cells": expected_counts.values,
    "observed_n_cells": observed_counts.reindex(expected_counts.index).fillna(0).astype(int).values,
})
count_audit["match"] = count_audit["expected_n_cells"].eq(count_audit["observed_n_cells"])

if len(main_cells) != EXPECTED_MAIN_N_CELLS:
    raise AssertionError(f"Expected {EXPECTED_MAIN_N_CELLS:,} main cells, observed {len(main_cells):,}.")
if main_cells["liana_celltype_main"].nunique() != EXPECTED_MAIN_N_CELLTYPES:
    raise AssertionError(
        f"Expected {EXPECTED_MAIN_N_CELLTYPES} main cell types, "
        f"observed {main_cells['liana_celltype_main'].nunique()}."
    )
if not count_audit["match"].all():
    raise AssertionError("Main cell-type counts do not match canonical Notebook 25 checkpoints.")

write_tsv_replace(count_audit, METADATA_DIR / f"{OUT_PREFIX}_celltype_counts_used.tsv")
display(count_audit)
print("[PASS] canonical main membership:", len(main_cells), "cells,", len(expected_types), "cell types")


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_celltype_counts_used.tsv (9, 4)


,liana_celltype,expected_n_cells,observed_n_cells,match
0,B_Plasma,1167,1167,True
1,Endothelial,2640,2640,True
2,Malignant_AC_like_reactive_malignant,22525,22525,True
3,Malignant_NPC_neuronal_like_malignant,6183,6183,True
4,Malignant_OPC_NPC_like_malignant,61646,61646,True
5,Malignant_cycling_candidate,38634,38634,True
6,Malignant_hypoxia_stress_malignant,32896,32896,True
7,Mural_Pericyte_Fibroblast,3566,3566,True
8,Oligodendrocyte_myelinating,14809,14809,True


[PASS] canonical main membership: 184066 cells, 9 cell types


In [7]:
# Read only H5AD obs/var indices directly from HDF5; do not instantiate the full canonical AnnData
obs_names = read_h5ad_axis_index(INPUT_H5AD, "obs")
var_names = read_h5ad_axis_index(INPUT_H5AD, "var")

input_n_obs = len(obs_names)
input_n_vars = len(var_names)

if input_n_obs != EXPECTED_INPUT_N_OBS or input_n_vars != EXPECTED_INPUT_N_VARS:
    raise AssertionError(
        f"Canonical H5AD shape mismatch from obs/var indices: observed {input_n_obs:,} x {input_n_vars:,}; "
        f"expected {EXPECTED_INPUT_N_OBS:,} x {EXPECTED_INPUT_N_VARS:,}."
    )
if not obs_names.is_unique:
    raise AssertionError("Canonical H5AD obs_names are not unique.")
if not var_names.is_unique:
    raise AssertionError("Canonical H5AD var_names are not unique; refusing silent var_names_make_unique().")

with h5py.File(INPUT_H5AD, "r") as h5:
    if f"layers/{COUNTS_LAYER}" not in h5:
        raise KeyError(f"Required counts layer {COUNTS_LAYER!r} is missing.")

requested_ids = pd.Index(main_cells["cell_id"].astype(str))
positions = obs_names.get_indexer(requested_ids)
missing_ids = requested_ids[positions < 0]

if len(missing_ids):
    raise AssertionError(
        f"{len(missing_ids)} canonical main cell IDs are absent from the input H5AD. "
        "Refusing silent intersection."
    )
if len(np.unique(positions)) != len(positions):
    raise AssertionError("Main-cell mapping to H5AD positions is not one-to-one.")

# Preserve canonical H5AD row order for deterministic matrix extraction.
row_positions = np.sort(positions.astype(np.int64))
selected_ids = obs_names[row_positions]

cell_to_label = main_cells.set_index("cell_id")["liana_celltype_main"]
selected_labels = cell_to_label.reindex(selected_ids)
if selected_labels.isna().any():
    raise AssertionError("NA LIANA labels after exact cell-ID reindexing.")

membership_audit = pd.DataFrame([
    {"item": "input_n_obs", "expected": EXPECTED_INPUT_N_OBS, "observed": input_n_obs, "match": input_n_obs == EXPECTED_INPUT_N_OBS},
    {"item": "input_n_vars", "expected": EXPECTED_INPUT_N_VARS, "observed": input_n_vars, "match": input_n_vars == EXPECTED_INPUT_N_VARS},
    {"item": "requested_main_cells", "expected": EXPECTED_MAIN_N_CELLS, "observed": len(requested_ids), "match": len(requested_ids) == EXPECTED_MAIN_N_CELLS},
    {"item": "matched_main_cells", "expected": EXPECTED_MAIN_N_CELLS, "observed": len(row_positions), "match": len(row_positions) == EXPECTED_MAIN_N_CELLS},
    {"item": "missing_main_cell_ids", "expected": 0, "observed": len(missing_ids), "match": len(missing_ids) == 0},
])
write_tsv_replace(membership_audit, METADATA_DIR / f"{OUT_PREFIX}_input_membership_audit.tsv")
display(membership_audit)

gc.collect()


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_input_membership_audit.tsv (5, 4)


,item,expected,observed,match
0,input_n_obs,351427,351427,True
1,input_n_vars,15176,15176,True
2,requested_main_cells,184066,184066,True
3,matched_main_cells,184066,184066,True
4,missing_main_cell_ids,0,0,True


33

In [8]:
# Preflight memory estimate, then stream only selected counts rows into RAM
layer_meta = inspect_csr_layer(INPUT_H5AD, COUNTS_LAYER, row_positions)

if tuple(layer_meta["shape"]) != (EXPECTED_INPUT_N_OBS, EXPECTED_INPUT_N_VARS):
    raise AssertionError(f"Counts-layer shape mismatch: {layer_meta['shape']}")

estimated_csr_gb = float(layer_meta["target_csr_gb"])
available_gb = available_ram_gb()
recommended_available_gb = max(6.0, 1.8 * estimated_csr_gb + 2.0)

memory_preflight = pd.DataFrame([
    {"item": "selected_counts_nnz", "value": layer_meta["selected_nnz"]},
    {"item": "source_counts_dtype", "value": layer_meta["source_data_dtype"]},
    {"item": "target_expression_dtype", "value": "float32"},
    {"item": "estimated_expression_csr_gb", "value": round(estimated_csr_gb, 3)},
    {"item": "available_ram_gb_before_load", "value": round(available_gb, 3) if pd.notna(available_gb) else np.nan},
    {"item": "recommended_available_ram_gb", "value": round(recommended_available_gb, 3)},
    {"item": "n_jobs_memory_safe", "value": N_JOBS},
])
write_tsv_replace(memory_preflight, METADATA_DIR / f"{OUT_PREFIX}_memory_preflight.tsv")
display(memory_preflight)

if pd.notna(available_gb) and available_gb < recommended_available_gb:
    raise MemoryError(
        f"Only {available_gb:.2f} GB RAM is currently available; "
        f"this run requires approximately >= {recommended_available_gb:.2f} GB available "
        "under the memory-safe configuration. Close other applications and rerun from a fresh kernel."
    )

counts_csr, layer_meta_loaded = load_selected_counts_csr(
    INPUT_H5AD,
    COUNTS_LAYER,
    row_positions,
    chunk_rows=H5_ROW_CHUNK,
)

if counts_csr.shape != (EXPECTED_MAIN_N_CELLS, EXPECTED_INPUT_N_VARS):
    raise AssertionError(f"Loaded counts shape mismatch: {counts_csr.shape}")
if counts_csr.nnz != EXPECTED_NORMALIZED_NNZ:
    raise AssertionError(
        f"Selected counts nnz={counts_csr.nnz:,}, expected historical/canonical {EXPECTED_NORMALIZED_NNZ:,}. "
        "Stop before LIANA because expression membership has drifted."
    )

print("[COUNTS LOADED]", counts_csr.shape, "nnz=", f"{counts_csr.nnz:,}", "dtype=", counts_csr.dtype)
print("[RAM available GB]", round(available_ram_gb(), 3) if PSUTIL_AVAILABLE else "not measured")


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_memory_preflight.tsv (7, 2)


,item,value
0,selected_counts_nnz,431978608
1,source_counts_dtype,int32
2,target_expression_dtype,float32
3,estimated_expression_csr_gb,3.22
4,available_ram_gb_before_load,8.153
5,recommended_available_ram_gb,7.796
6,n_jobs_memory_safe,1


[COUNTS LOADED] (184066, 15176) nnz= 431,978,608 dtype= float32
[RAM available GB] 5.1


In [9]:
# Build minimal AnnData and reproduce historical normalize_total(1e4) -> log1p semantics in-place
gene_nonzero_cells = np.bincount(counts_csr.indices, minlength=counts_csr.shape[1])
n_zero_genes = int(np.sum(gene_nonzero_cells == 0))
if n_zero_genes != 0:
    raise AssertionError(
        f"Historical sc.pp.filter_genes(min_cells=1) would remove {n_zero_genes} genes; "
        "expected 0 for the canonical 184,066-cell subset."
    )

normalize_log1p_csr_inplace(
    counts_csr,
    target_sum=NORMALIZATION_TARGET_SUM,
    row_chunk=2048,
)

normalized_nnz = int(counts_csr.nnz)
if normalized_nnz != EXPECTED_NORMALIZED_NNZ:
    raise AssertionError(
        f"Normalized X nnz={normalized_nnz:,}; expected {EXPECTED_NORMALIZED_NNZ:,}."
    )
if np.any(~np.isfinite(counts_csr.data)):
    raise ValueError("Non-finite values found after normalization/log1p.")

obs_min = pd.DataFrame(
    {GROUPBY: pd.Categorical(selected_labels.astype(str).values)},
    index=selected_ids,
)
var_min = pd.DataFrame(index=var_names)

adata_liana = ad.AnnData(
    X=counts_csr,
    obs=obs_min,
    var=var_min,
)
adata_liana.raw = None

del counts_csr, obs_min, var_min, gene_nonzero_cells
gc.collect()

expression_audit = pd.DataFrame([
    {"item": "n_cells", "expected": EXPECTED_MAIN_N_CELLS, "observed": adata_liana.n_obs, "match": adata_liana.n_obs == EXPECTED_MAIN_N_CELLS},
    {"item": "n_genes_after_historical_min_cells_1", "expected": EXPECTED_INPUT_N_VARS, "observed": adata_liana.n_vars, "match": adata_liana.n_vars == EXPECTED_INPUT_N_VARS},
    {"item": "normalized_X_nnz", "expected": EXPECTED_NORMALIZED_NNZ, "observed": normalized_nnz, "match": normalized_nnz == EXPECTED_NORMALIZED_NNZ},
    {"item": "X_dtype", "expected": "float32", "observed": str(adata_liana.X.dtype), "match": str(adata_liana.X.dtype) == "float32"},
    {"item": "normalization_target_sum", "expected": NORMALIZATION_TARGET_SUM, "observed": NORMALIZATION_TARGET_SUM, "match": True},
    {"item": "transform", "expected": "log1p", "observed": "log1p", "match": True},
])
write_tsv_replace(expression_audit, METADATA_DIR / f"{OUT_PREFIX}_expression_audit.tsv")
display(expression_audit)

print("[LIANA minimal object]", adata_liana)
print("[RAM available GB]", round(available_ram_gb(), 3) if PSUTIL_AVAILABLE else "not measured")


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_expression_audit.tsv (6, 4)


,item,expected,observed,match
0,n_cells,184066,184066,True
1,n_genes_after_historical_min_cells_1,15176,15176,True
2,normalized_X_nnz,431978608,431978608,True
3,X_dtype,float32,float32,True
4,normalization_target_sum,10000.0,10000.0,True
5,transform,log1p,log1p,True


[LIANA minimal object] AnnData object with n_obs × n_vars = 184066 × 15176
    obs: 'liana_celltype_global_main'
[RAM available GB] 5.405


In [10]:
# Run canonical global LIANA
run_config = pd.DataFrame([
    {"parameter": "groupby", "value": GROUPBY},
    {"parameter": "resource_name", "value": RESOURCE_NAME},
    {"parameter": "expr_prop", "value": EXPR_PROP},
    {"parameter": "min_cells", "value": MIN_CELLS},
    {"parameter": "n_perms", "value": str(N_PERMS)},
    {"parameter": "seed", "value": SEED},
    {"parameter": "n_jobs", "value": N_JOBS},
    {"parameter": "historical_n_jobs", "value": HISTORICAL_N_JOBS},
    {"parameter": "n_jobs_change_reason", "value": "memory_safety_only"},
    {"parameter": "liana_version", "value": LIANA_VERSION},
    {"parameter": "counts_layer_source", "value": COUNTS_LAYER},
    {"parameter": "normalization_target_sum", "value": NORMALIZATION_TARGET_SUM},
    {"parameter": "normalization_transform", "value": "log1p"},
    {"parameter": "use_raw", "value": False},
    {"parameter": "layer_passed_to_liana", "value": "None"},
    {"parameter": "expression_dtype", "value": str(adata_liana.X.dtype)},
    {"parameter": "n_cells_used", "value": adata_liana.n_obs},
    {"parameter": "n_genes_used", "value": adata_liana.n_vars},
    {"parameter": "expected_primary_rank", "value": EXPECTED_PRIMARY_RANK},
    {"parameter": "rank_threshold", "value": RANK_THRESHOLD},
    {"parameter": "created_at", "value": datetime.now().isoformat(timespec="seconds")},
])
write_tsv_replace(run_config, METADATA_DIR / f"{OUT_PREFIX}_run_config.tsv")
display(run_config)

li.mt.rank_aggregate(
    adata_liana,
    groupby=GROUPBY,
    resource_name=RESOURCE_NAME,
    expr_prop=EXPR_PROP,
    min_cells=MIN_CELLS,
    use_raw=False,
    layer=None,
    n_perms=N_PERMS,
    n_jobs=N_JOBS,
    seed=SEED,
    verbose=True,
    inplace=True,
    key_added="liana_res",
)

if "liana_res" not in adata_liana.uns:
    raise RuntimeError("LIANA completed without adata.uns['liana_res'].")

res = adata_liana.uns["liana_res"].copy()
n_cells_used = int(adata_liana.n_obs)
n_genes_used = int(adata_liana.n_vars)
n_celltypes_used = int(adata_liana.obs[GROUPBY].nunique())

print("[LIANA result]", res.shape)

del adata_liana, selected_labels, selected_ids, cell_to_label
gc.collect()
print("[RAM available GB after releasing expression]", round(available_ram_gb(), 3) if PSUTIL_AVAILABLE else "not measured")

display(res.head())


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_run_config.tsv (21, 2)


,parameter,value
0,groupby,liana_celltype_global_main
1,resource_name,consensus
2,expr_prop,0.1
3,min_cells,50
4,n_perms,None
5,seed,1337
6,n_jobs,1
7,historical_n_jobs,4
8,n_jobs_change_reason,memory_safety_only
9,liana_version,1.7.1


Using resource `consensus`.
Using `.X`!
0.29 of entities in the resource are missing from the data.


Generating ligand-receptor stats for 184066 samples and 1210 features
Assuming that counts were `natural` log-normalized!
Running CellPhoneDB
Running Connectome
Running log2FC
Running NATMI
Running SingleCellSignalR
[LIANA result] (23797, 11)
[RAM available GB after releasing expression] 10.123


,source,target,ligand_complex,receptor_complex,lr_means,expr_prod,scaled_weight,lr_logfc,spec_weight,lrscore,magnitude_rank
13640,Malignant_cycling_candidate,Malignant_OPC_NPC_like_malignant,PTN,PTPRZ1,3.150313,9.909274,0.493709,0.826383,0.048963,0.943303,1.589294e-08
10826,Malignant_OPC_NPC_like_malignant,Malignant_OPC_NPC_like_malignant,PTN,PTPRZ1,3.120315,9.727659,0.471571,0.866426,0.048065,0.942806,6.356997e-08
21285,Mural_Pericyte_Fibroblast,Mural_Pericyte_Fibroblast,TIMP1,CD63,3.060620,9.249804,0.883139,1.765574,0.038766,0.941433,1.430284e-07
19761,Mural_Pericyte_Fibroblast,Malignant_OPC_NPC_like_malignant,TIMP1,CD63,3.009802,8.903878,0.836792,1.877996,0.037316,0.940373,2.542656e-07
21683,Mural_Pericyte_Fibroblast,Oligodendrocyte_myelinating,TIMP1,CD63,2.977001,8.680599,0.806861,1.734864,0.036380,0.939657,7.786231e-07


In [11]:
# Strict result schema, canonical label semantics, and fixed rank selection
required_result_cols = {
    "source", "target", "ligand_complex", "receptor_complex", EXPECTED_PRIMARY_RANK
}
missing_result_cols = required_result_cols - set(res.columns)
if missing_result_cols:
    raise ValueError(
        f"LIANA result schema drift. Missing exact required columns: {sorted(missing_result_cols)}"
    )

primary_rank = EXPECTED_PRIMARY_RANK

if res["source"].isna().any() or res["target"].isna().any():
    raise ValueError("NA source/target labels in LIANA results.")
if res["ligand_complex"].isna().any() or res["receptor_complex"].isna().any():
    raise ValueError("NA ligand/receptor complex labels in LIANA results.")

observed_result_celltypes = set(res["source"].astype(str)).union(set(res["target"].astype(str)))
expected_result_celltypes = set(EXPECTED_MAIN_COUNTS)

if observed_result_celltypes != expected_result_celltypes:
    raise AssertionError(
        "LIANA result source/target labels differ from the 9 canonical main cell types.\n"
        f"Missing: {sorted(expected_result_celltypes - observed_result_celltypes)}\n"
        f"Unexpected: {sorted(observed_result_celltypes - expected_result_celltypes)}"
    )

if any("Malignant_non_malignant" in x for x in observed_result_celltypes):
    raise AssertionError("Historical malformed Malignant_non_malignant label survived into Notebook 26.")

res["primary_rank_col"] = primary_rank
res["primary_rank"] = pd.to_numeric(res[primary_rank], errors="coerce")
if res["primary_rank"].isna().any():
    raise ValueError("Non-numeric/NA magnitude_rank values detected.")

res["lr_pair"] = res["ligand_complex"].astype(str) + "→" + res["receptor_complex"].astype(str)
res["source_broad_group"] = res["source"].map(broad_group)
res["target_broad_group"] = res["target"].map(broad_group)
res["communication_axis"] = res["source_broad_group"] + "_to_" + res["target_broad_group"]

res = res.sort_values(
    ["primary_rank", "source", "target", "ligand_complex", "receptor_complex"],
    ascending=[True, True, True, True, True],
).reset_index(drop=True)

full_path = METADATA_DIR / f"{OUT_PREFIX}_all_results.tsv.gz"
if full_path.exists():
    full_path.unlink()
res.to_csv(full_path, sep="\t", index=False, compression="gzip")
print("[WRITE]", full_path, res.shape)

top_global = res.head(TOP_N_GLOBAL).copy()
write_tsv_replace(top_global, METADATA_DIR / f"{OUT_PREFIX}_top_global_interactions.tsv")

rank_sig = res.loc[res["primary_rank"] <= RANK_THRESHOLD].copy()
rank_sig["selection_rule"] = f"primary_rank<={RANK_THRESHOLD}"
write_tsv_replace(
    rank_sig,
    METADATA_DIR / f"{OUT_PREFIX}_rank_threshold_or_top_interactions.tsv",
)

print("[rank selection]", rank_sig.shape, "fixed threshold:", RANK_THRESHOLD)
display(top_global.head(20))


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_all_results.tsv.gz (23797, 17)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_top_global_interactions.tsv (1000, 17)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_rank_threshold_or_top_interactions.tsv (1384, 18)
[rank selection] (1384, 18) fixed threshold: 0.05


,source,target,ligand_complex,receptor_complex,lr_means,expr_prod,scaled_weight,lr_logfc,spec_weight,lrscore,magnitude_rank,primary_rank_col,primary_rank,lr_pair,source_broad_group,target_broad_group,communication_axis
0,Malignant_cycling_candidate,Malignant_OPC_NPC_like_malignant,PTN,PTPRZ1,3.150313,9.909274,0.493709,0.826383,0.048963,0.943303,1.589294e-08,magnitude_rank,1.589294e-08,PTN→PTPRZ1,malignant_or_cycling,malignant_or_cycling,malignant_or_cycling_to_malignant_or_cycling
1,Malignant_OPC_NPC_like_malignant,Malignant_OPC_NPC_like_malignant,PTN,PTPRZ1,3.120315,9.727659,0.471571,0.866426,0.048065,0.942806,6.356997e-08,magnitude_rank,6.356997e-08,PTN→PTPRZ1,malignant_or_cycling,malignant_or_cycling,malignant_or_cycling_to_malignant_or_cycling
2,Mural_Pericyte_Fibroblast,Mural_Pericyte_Fibroblast,TIMP1,CD63,3.060620,9.249804,0.883139,1.765574,0.038766,0.941433,1.430284e-07,magnitude_rank,1.430284e-07,TIMP1→CD63,non_malignant_reference_like,non_malignant_reference_like,non_malignant_reference_like_to_non_malignant_...
3,Mural_Pericyte_Fibroblast,Malignant_OPC_NPC_like_malignant,TIMP1,CD63,3.009802,8.903878,0.836792,1.877996,0.037316,0.940373,2.542656e-07,magnitude_rank,2.542656e-07,TIMP1→CD63,non_malignant_reference_like,malignant_or_cycling,non_malignant_reference_like_to_malignant_or_c...
4,Malignant_cycling_candidate,Malignant_cycling_candidate,PTN,PTPRZ1,2.970492,8.731945,0.371164,0.397266,0.043146,0.939824,7.786231e-07,magnitude_rank,7.786231e-07,PTN→PTPRZ1,malignant_or_cycling,malignant_or_cycling,malignant_or_cycling_to_malignant_or_cycling
5,Mural_Pericyte_Fibroblast,Oligodendrocyte_myelinating,TIMP1,CD63,2.977001,8.680599,0.806861,1.734864,0.036380,0.939657,7.786231e-07,magnitude_rank,7.786231e-07,TIMP1→CD63,non_malignant_reference_like,non_malignant_reference_like,non_malignant_reference_like_to_non_malignant_...
6,Malignant_AC_like_reactive_malignant,Malignant_OPC_NPC_like_malignant,PTN,PTPRZ1,2.966972,8.799319,0.358488,0.711524,0.043478,0.940041,1.016949e-06,magnitude_rank,1.016949e-06,PTN→PTPRZ1,malignant_or_cycling,malignant_or_cycling,malignant_or_cycling_to_malignant_or_cycling
7,Mural_Pericyte_Fibroblast,Malignant_AC_like_reactive_malignant,TIMP1,CD63,2.974782,8.665500,0.804840,1.737468,0.036317,0.939608,1.016949e-06,magnitude_rank,1.016949e-06,TIMP1→CD63,non_malignant_reference_like,malignant_or_cycling,non_malignant_reference_like_to_malignant_or_c...
8,Malignant_OPC_NPC_like_malignant,Malignant_cycling_candidate,PTN,PTPRZ1,2.940493,8.571909,0.349027,0.437309,0.042355,0.939299,1.588893e-06,magnitude_rank,1.588893e-06,PTN→PTPRZ1,malignant_or_cycling,malignant_or_cycling,malignant_or_cycling_to_malignant_or_cycling
9,Mural_Pericyte_Fibroblast,Malignant_cycling_candidate,TIMP1,CD63,2.941979,8.442204,0.774921,1.717841,0.035381,0.938863,1.588893e-06,magnitude_rank,1.588893e-06,TIMP1→CD63,non_malignant_reference_like,malignant_or_cycling,non_malignant_reference_like_to_malignant_or_c...


In [12]:
# Source-target, communication-axis, and historical checkpoint audits
source_target_summary = (
    rank_sig
    .groupby(
        ["source", "target", "source_broad_group", "target_broad_group", "communication_axis"],
        observed=True,
    )
    .agg(
        n_prioritized_interactions=("lr_pair", "nunique"),
        median_primary_rank=("primary_rank", "median"),
        min_primary_rank=("primary_rank", "min"),
    )
    .reset_index()
    .sort_values(["n_prioritized_interactions", "median_primary_rank"], ascending=[False, True])
)
write_tsv_replace(
    source_target_summary,
    METADATA_DIR / f"{OUT_PREFIX}_source_target_summary.tsv",
)

axis_summary = (
    rank_sig
    .groupby("communication_axis", observed=True)
    .agg(
        n_prioritized_interactions=("lr_pair", "nunique"),
        n_source_target_pairs=("source", lambda x: len(set(zip(
            rank_sig.loc[x.index, "source"],
            rank_sig.loc[x.index, "target"],
        )))),
        median_primary_rank=("primary_rank", "median"),
        min_primary_rank=("primary_rank", "min"),
    )
    .reset_index()
    .sort_values(["n_prioritized_interactions", "median_primary_rank"], ascending=[False, True])
)
write_tsv_replace(axis_summary, METADATA_DIR / f"{OUT_PREFIX}_axis_summary.tsv")

observed_axes = set(axis_summary["communication_axis"].astype(str))
if observed_axes != EXPECTED_COMMUNICATION_AXES:
    raise AssertionError(
        "Canonical corrected labels should produce all four communication axes.\n"
        f"Missing: {sorted(EXPECTED_COMMUNICATION_AXES - observed_axes)}\n"
        f"Unexpected: {sorted(observed_axes - EXPECTED_COMMUNICATION_AXES)}"
    )

n_unique_lr_pairs = int(rank_sig["lr_pair"].nunique())
n_source_target_pairs = int(len(set(zip(rank_sig["source"], rank_sig["target"]))))

historical_audit = pd.DataFrame([
    {"checkpoint": "n_liana_results", "expected_historical": HISTORICAL_RESULT_CHECKPOINTS["n_liana_results"], "observed": len(res)},
    {"checkpoint": "n_prioritized", "expected_historical": HISTORICAL_RESULT_CHECKPOINTS["n_prioritized"], "observed": len(rank_sig)},
    {"checkpoint": "n_unique_lr_pairs", "expected_historical": HISTORICAL_RESULT_CHECKPOINTS["n_unique_lr_pairs"], "observed": n_unique_lr_pairs},
    {"checkpoint": "n_source_target_pairs", "expected_historical": HISTORICAL_RESULT_CHECKPOINTS["n_source_target_pairs"], "observed": n_source_target_pairs},
])
historical_audit["match"] = historical_audit["expected_historical"].eq(historical_audit["observed"])

write_tsv_replace(
    historical_audit,
    METADATA_DIR / f"{OUT_PREFIX}_historical_result_checkpoint_audit.tsv",
)

print("[Historical checkpoints are audit-only; they do not alter current results.]")
display(historical_audit)
display(axis_summary)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_source_target_summary.tsv (81, 8)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_axis_summary.tsv (4, 5)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_historical_result_checkpoint_audit.tsv (4, 4)
[Historical checkpoints are audit-only; they do not alter current results.]


,checkpoint,expected_historical,observed,match
0,n_liana_results,23797,23797,True
1,n_prioritized,1384,1384,True
2,n_unique_lr_pairs,264,264,True
3,n_source_target_pairs,81,81,True


,communication_axis,n_prioritized_interactions,n_source_target_pairs,median_primary_rank,min_primary_rank
3,non_malignant_reference_like_to_non_malignant_...,168,16,0.015616,1.430284e-07
2,non_malignant_reference_like_to_malignant_or_c...,98,20,0.017271,2.542656e-07
1,malignant_or_cycling_to_non_malignant_referenc...,88,20,0.017740,1.073610e-05
0,malignant_or_cycling_to_malignant_or_cycling,60,25,0.012463,1.589294e-08


In [13]:
# Detailed canonical summaries
top_per_pair = (
    res
    .groupby(["source", "target"], observed=True, group_keys=False)
    .head(TOP_N_PER_SOURCE_TARGET)
    .reset_index(drop=True)
)
write_tsv_replace(
    top_per_pair,
    METADATA_DIR / f"{OUT_PREFIX}_top_source_target_interactions.tsv",
)

malignant_related = rank_sig.loc[
    rank_sig["source_broad_group"].eq("malignant_or_cycling")
    | rank_sig["target_broad_group"].eq("malignant_or_cycling")
].copy()
write_tsv_replace(
    malignant_related,
    METADATA_DIR / f"{OUT_PREFIX}_prioritized_malignant_related_interactions.tsv",
)

lr_summary = (
    rank_sig
    .groupby(["ligand_complex", "receptor_complex", "lr_pair"], observed=True)
    .agg(
        n_source_target_pairs=("source", lambda x: len(set(zip(
            rank_sig.loc[x.index, "source"],
            rank_sig.loc[x.index, "target"],
        )))),
        n_sources=("source", "nunique"),
        n_targets=("target", "nunique"),
        median_primary_rank=("primary_rank", "median"),
        min_primary_rank=("primary_rank", "min"),
    )
    .reset_index()
    .sort_values(["n_source_target_pairs", "median_primary_rank"], ascending=[False, True])
)
write_tsv_replace(
    lr_summary,
    METADATA_DIR / f"{OUT_PREFIX}_ligand_receptor_summary.tsv",
)

rank_sig["biological_axis"] = rank_sig.apply(label_axis, axis=1)
rank_sig["target_lineage_simplified"] = rank_sig["target"].map(target_lineage)
rank_sig["source_lineage_simplified"] = rank_sig["source"].map(target_lineage)

bio_summary = (
    rank_sig
    .groupby(
        ["biological_axis", "source_lineage_simplified", "target_lineage_simplified"],
        observed=True,
    )
    .agg(
        n_interactions=("lr_pair", "nunique"),
        n_source_target_pairs=("source", lambda x: len(set(zip(
            rank_sig.loc[x.index, "source"],
            rank_sig.loc[x.index, "target"],
        )))),
        median_primary_rank=("primary_rank", "median"),
        min_primary_rank=("primary_rank", "min"),
    )
    .reset_index()
    .sort_values(["n_interactions", "median_primary_rank"], ascending=[False, True])
)
write_tsv_replace(
    bio_summary,
    METADATA_DIR / f"{OUT_PREFIX}_biological_axis_summary.tsv",
)

write_tsv_replace(
    rank_sig,
    METADATA_DIR / f"{OUT_PREFIX}_prioritized_interactions_annotated.tsv",
)

display(lr_summary.head(30))
display(bio_summary.head(30))


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_top_source_target_interactions.tsv (2430, 17)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_prioritized_malignant_related_interactions.tsv (1044, 18)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_ligand_receptor_summary.tsv (264, 8)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_biological_axis_summary.tsv (25, 7)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_prioritized_interactions_annotated.tsv (1384, 21)


,ligand_complex,receptor_complex,lr_pair,n_source_target_pairs,n_sources,n_targets,median_primary_rank,min_primary_rank
241,TIMP1,CD63,TIMP1→CD63,68,9,9,0.000642,1.430284e-07
153,HLA-A,APLP2,HLA-A→APLP2,64,9,8,0.007831,3.608520e-04
208,PTN,NCL,PTN→NCL,63,7,9,0.004415,1.245064e-05
42,CALM1,PTPRA,CALM1→PTPRA,53,7,8,0.015115,3.763617e-03
212,PTN,PTPRZ1,PTN→PTPRZ1,49,8,8,0.000324,1.589294e-08
174,LGALS1,ITGB1,LGALS1→ITGB1,42,9,9,0.009852,4.066882e-06
185,MIF,EGFR,MIF→EGFR,37,9,6,0.016187,1.091229e-03
260,VIM,CD44,VIM→CD44,30,7,5,0.003378,2.131663e-04
182,MDK,PTPRZ1,MDK→PTPRZ1,27,8,5,0.003094,5.155514e-05
215,PTN,SDC3,PTN→SDC3,27,7,5,0.014607,7.714426e-04


,biological_axis,source_lineage_simplified,target_lineage_simplified,n_interactions,n_source_target_pairs,median_primary_rank,min_primary_rank
7,microenvironment_to_malignant,mural_pericyte_fibroblast,malignant_state,69,5,0.013717,2.542656e-07
19,microenvironment_to_microenvironment,mural_pericyte_fibroblast,mural_pericyte_fibroblast,63,1,0.013817,1.430284e-07
0,malignant_state_to_malignant_state,malignant_state,malignant_state,60,25,0.012463,1.589294e-08
18,microenvironment_to_microenvironment,mural_pericyte_fibroblast,endothelial,54,1,0.017783,5.146859e-06
3,malignant_to_microenvironment,malignant_state,mural_pericyte_fibroblast,43,5,0.017740,2.668609e-05
2,malignant_to_microenvironment,malignant_state,endothelial,38,5,0.019379,8.685173e-05
6,microenvironment_to_malignant,endothelial,malignant_state,36,5,0.025268,3.513761e-04
14,microenvironment_to_microenvironment,endothelial,endothelial,30,1,0.022652,1.157751e-05
20,microenvironment_to_microenvironment,mural_pericyte_fibroblast,oligodendrocyte_reference,29,1,0.021864,7.786231e-07
17,microenvironment_to_microenvironment,mural_pericyte_fibroblast,B_plasma,27,1,0.007378,4.291310e-05


In [14]:
# Diagnostic figures — generated for historical continuity; do NOT stage during Main Analysis phase
figure_records = []

pivot_counts = (
    source_target_summary
    .pivot(index="source", columns="target", values="n_prioritized_interactions")
    .fillna(0)
)
row_order = pivot_counts.sum(axis=1).sort_values(ascending=False).index
col_order = pivot_counts.sum(axis=0).sort_values(ascending=False).index
pivot_counts = pivot_counts.loc[row_order, col_order]

plt.figure(figsize=(max(10, 0.55 * pivot_counts.shape[1]), max(8, 0.45 * pivot_counts.shape[0])))
plt.imshow(pivot_counts.values, aspect="auto")
plt.colorbar(label="Number of prioritized LR interactions")
plt.xticks(range(pivot_counts.shape[1]), pivot_counts.columns, rotation=90, fontsize=8)
plt.yticks(range(pivot_counts.shape[0]), pivot_counts.index, fontsize=8)
plt.title("LIANA global main: prioritized LR interaction counts")
plt.xlabel("Target cell type")
plt.ylabel("Source cell type")
fig_path = savefig(FIG_DIR / f"{OUT_PREFIX}_source_target_prioritized_count_heatmap.png")
figure_records.append({
    "figure": str(fig_path),
    "description": "Diagnostic heatmap of prioritized LIANA LR interaction counts by source-target pair.",
})

pivot_rank = (
    source_target_summary
    .pivot(index="source", columns="target", values="median_primary_rank")
    .reindex(index=row_order, columns=col_order)
)
plt.figure(figsize=(max(10, 0.55 * pivot_rank.shape[1]), max(8, 0.45 * pivot_rank.shape[0])))
plt.imshow(pivot_rank.values, aspect="auto")
plt.colorbar(label=f"Median {primary_rank} among prioritized interactions")
plt.xticks(range(pivot_rank.shape[1]), pivot_rank.columns, rotation=90, fontsize=8)
plt.yticks(range(pivot_rank.shape[0]), pivot_rank.index, fontsize=8)
plt.title("LIANA global main: median priority rank")
plt.xlabel("Target cell type")
plt.ylabel("Source cell type")
fig_path = savefig(FIG_DIR / f"{OUT_PREFIX}_source_target_median_rank_heatmap.png")
figure_records.append({
    "figure": str(fig_path),
    "description": "Diagnostic heatmap of median LIANA primary rank by source-target pair.",
})

plt.figure(figsize=(10, 5))
axdf = axis_summary.sort_values("n_prioritized_interactions", ascending=True)
plt.barh(axdf["communication_axis"], axdf["n_prioritized_interactions"])
plt.xlabel("Number of prioritized LR interactions")
plt.ylabel("Communication axis")
plt.title("LIANA global main: corrected broad communication axes")
fig_path = savefig(FIG_DIR / f"{OUT_PREFIX}_axis_summary_barplot.png")
figure_records.append({
    "figure": str(fig_path),
    "description": "Diagnostic barplot of corrected malignant/non-malignant communication axes.",
})

figure_manifest = pd.DataFrame(figure_records)
write_tsv_replace(
    figure_manifest,
    METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv",
)
display(figure_manifest)


[FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\LIANA_HVG8000_r0_6\LIANA_global_main_HVG8000_r0_6_source_target_prioritized_count_heatmap.png
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\LIANA_HVG8000_r0_6\LIANA_global_main_HVG8000_r0_6_source_target_median_rank_heatmap.png
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\LIANA_HVG8000_r0_6\LIANA_global_main_HVG8000_r0_6_axis_summary_barplot.png
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_figure_manifest.tsv (3, 2)


,figure,description
0,G:\Repository\glioma-cnv-single-cell-atlas\fig...,Diagnostic heatmap of prioritized LIANA LR int...
1,G:\Repository\glioma-cnv-single-cell-atlas\fig...,Diagnostic heatmap of median LIANA primary ran...
2,G:\Repository\glioma-cnv-single-cell-atlas\fig...,Diagnostic barplot of corrected malignant/non-...


In [15]:
# Final report and output manifest
summary_records = [
    {"item": "n_cells_used", "value": n_cells_used},
    {"item": "n_genes_used", "value": n_genes_used},
    {"item": "n_liana_celltypes", "value": n_celltypes_used},
    {"item": "n_liana_results", "value": len(res)},
    {"item": "primary_rank_col", "value": primary_rank},
    {"item": "n_top_global_saved", "value": min(TOP_N_GLOBAL, len(res))},
    {"item": "n_prioritized_or_rank_threshold_interactions", "value": len(rank_sig)},
    {"item": "rank_threshold_used", "value": RANK_THRESHOLD},
    {"item": "n_unique_lr_pairs_prioritized", "value": n_unique_lr_pairs},
    {"item": "n_source_target_pairs_prioritized", "value": n_source_target_pairs},
    {"item": "n_communication_axes", "value": len(observed_axes)},
    {"item": "resource_name", "value": RESOURCE_NAME},
    {"item": "expr_prop", "value": EXPR_PROP},
    {"item": "min_cells", "value": MIN_CELLS},
    {"item": "n_perms", "value": str(N_PERMS)},
    {"item": "seed", "value": SEED},
    {"item": "n_jobs", "value": N_JOBS},
    {"item": "liana_version", "value": LIANA_VERSION},
    {"item": "memory_safe_streamed_counts", "value": True},
    {"item": "historical_result_checkpoints_all_match", "value": bool(historical_audit["match"].all())},
    {"item": "completed_at", "value": datetime.now().isoformat(timespec="seconds")},
]
summary = pd.DataFrame(summary_records)
write_tsv_replace(summary, METADATA_DIR / f"{OUT_PREFIX}_run_summary.tsv")
display(summary)

manifest_paths = [
    METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_celltype_counts_used.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_input_membership_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_memory_preflight.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_expression_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_run_config.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_all_results.tsv.gz",
    METADATA_DIR / f"{OUT_PREFIX}_top_global_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_rank_threshold_or_top_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_top_source_target_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_source_target_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_axis_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_ligand_receptor_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_biological_axis_summary.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_prioritized_malignant_related_interactions.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_prioritized_interactions_annotated.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_historical_result_checkpoint_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_run_summary.tsv",
]
manifest = pd.DataFrame([
    {
        "path": str(p),
        "exists": p.exists(),
        "size_mb": round(p.stat().st_size / 1024 / 1024, 3) if p.exists() else np.nan,
    }
    for p in manifest_paths
])
write_tsv_replace(
    manifest,
    METADATA_DIR / f"{OUT_PREFIX}_output_manifest.tsv",
)
display(manifest)

if not manifest["exists"].all():
    raise AssertionError("One or more expected Notebook 26 metadata outputs are missing.")

gc.collect()
print("[DONE] Notebook 26 completed.")


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_run_summary.tsv (21, 2)


,item,value
0,n_cells_used,184066
1,n_genes_used,15176
2,n_liana_celltypes,9
3,n_liana_results,23797
4,primary_rank_col,magnitude_rank
5,n_top_global_saved,1000
6,n_prioritized_or_rank_threshold_interactions,1384
7,rank_threshold_used,0.05
8,n_unique_lr_pairs_prioritized,264
9,n_source_target_pairs_prioritized,81


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\LIANA_global_main_HVG8000_r0_6_output_manifest.tsv (19, 3)


,path,exists,size_mb
0,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
1,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
2,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
3,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
4,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
5,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
6,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,1.456
7,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.307
8,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.449
9,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.740


[DONE] Notebook 26 completed.


## Interpretation guardrails

- This is the **global pooled** LIANA main analysis, not the condition-specific analysis.
- `magnitude_rank` is the fixed primary rank; lower values indicate higher priority.
- The prioritized set is defined strictly by `magnitude_rank <= 0.05`.
- Historical result counts are recorded only as reproducibility checkpoints; they never modify current results.
- Canonical Notebook 25 labels are authoritative. No `24b` post-hoc label repair should be rerun.
- Diagnostic figures are generated but remain parked until the later Figure/Supplementary phase.
